In [1]:
import pyterrier as pt
import pandas as pd


dataset = pt.get_dataset("msmarco_passage")
# print(f'Dataset: \n{dataset}')
index = dataset.get_index(variant="terrier_stemmed")

# Load DL-hard annotations
annotations_url = "https://raw.githubusercontent.com/grill-lab/DL-Hard/main/annotations/query/annotations.tsv"
annotations = pd.read_csv(
    annotations_url,
    sep="\t",
    header=None,
    names=["qid", "query", "intent", "answer_type", "topic_domain", "serp_type"],
)
annotations["qid"] = annotations["qid"].astype(str)

# Load TREC DL 2019/2020 qrels. This dataset contains relevance judgements from scale 0 to 3.
dl19 = pt.get_dataset("irds:msmarco-passage/trec-dl-2019")
dl20 = pt.get_dataset("irds:msmarco-passage/trec-dl-2020")
all_qrels = pd.concat([dl19.get_qrels(), dl20.get_qrels()])
all_qrels["qid"] = all_qrels["qid"].astype(str)

# Keep only queries that have both annotations and judgements.
qrels_qids = set(all_qrels["qid"].unique())
topics = annotations[annotations["qid"].isin(qrels_qids)].reset_index(drop=True)

print(f'qrels: \n{all_qrels}')

print(f"Queries with both annotations and relevance values: {len(topics)}")
print(topics["intent"].value_counts())
print("\n")
print(topics["answer_type"].value_counts())
topics["query_length"] = topics["query"].str.len()
max_len = topics["query_length"].max()
bins = range(0, max_len + 10, 10)

topics["query_length_group"] = pd.cut(topics["query_length"], bins=bins, labels=[f"{i+1}-{i+10}" for i in bins][:-1])
print(f"topics: \n{topics}")

qrels: 
           qid    docno  label iteration
0        19335  1017759      0        Q0
1        19335  1082489      0        Q0
2        19335   109063      0        Q0
3        19335  1160863      0        Q0
4        19335  1160871      0        Q0
...        ...      ...    ...       ...
11381  1136962  8526087      0         0
11382  1136962  8537921      0         0
11383  1136962  8742482      0         0
11384  1136962   937258      1         0
11385  1136962   999215      0         0

[20646 rows x 4 columns]
Queries with both annotations and relevance values: 97
intent
description     48
list            10
quantity         9
reason           8
attribute        7
entity           5
verification     3
process          3
language         3
weather          1
Name: count, dtype: int64


answer_type
factoid              28
definition           21
long answer          16
list                 10
short description     7
short answer          5
comparison            5
guide         

In [2]:
import re

def clean_query(q):
    q = re.sub(r"[^\w\s]", " ", str(q))
    q = re.sub(r"\s+", " ", q).strip()
    return q

# Clean queries (else it gives errors in PyTerrier)
topics["query"] = topics["query"].apply(clean_query)

# Initialize BM25 retriever
bm25 = pt.terrier.Retriever(index, wmodel="BM25")

# Initialize classic query expansion models
rm3 = pt.rewrite.RM3(index)
bo1 = pt.rewrite.Bo1QueryExpansion(index)

bm25_rm3 = bm25 >> rm3 >> bm25
bm25_bo1 = bm25 >> bo1 >> bm25

Java started (triggered by Retriever.__init__) and loaded: pyterrier.java, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]


In [ ]:
from pyterrier.measures import RR, nDCG, AP, R

results_classical = pt.Experiment(
    [bm25, bm25_rm3, bm25_bo1],
    topics[["qid", "query"]],  
    all_qrels,
    eval_metrics=[nDCG @ 10, AP(rel=2), RR(rel=2) @ 10, R(rel=2) @ 100],
    names=["BM25", "RM3", "Bo1"],
    perquery=True,
)

results = pd.concat([results_classical], ignore_index=True)

label_map = topics[["qid", "intent", "query_length_group", "meta_answer_type"]]
results = results.merge(label_map, on="qid", how="left")

# Overall results
print("\n=== Overall Results ===")
overall = results.groupby(["name", "measure"])["value"].mean().unstack()
print(overall)

# Results by query length
print("\n=== nDCG@10 by Query Length ===")
ndcg_by_length = (
    results[results["measure"] == "nDCG@10"]
    .groupby(["name", "query_length_group"])["value"]
    .mean()
    .unstack()
)
print(ndcg_by_length)
